## Testing Llama-2

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
from langchain_community.document_loaders import PyPDFLoader
import pypdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
# load model and tokenizer
model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="flex_attention",
    device_map="cpu"
)


Loading checkpoint shards: 100%|██████████| 2/2 [00:42<00:00, 21.46s/it]


In [23]:
# read document

loader = PyPDFLoader(r"C:\Users\rpurohit\Documents\GitHub\llm-benchmarking\data\temp\new-haven-zoning-code.pdf")
pages = loader.load_and_split()

In [24]:
# chunk document
chunk_size_value = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size_value,
                                                chunk_overlap=chunk_overlap,
                                                length_function=len)

split_text = text_splitter.split_documents(pages)

In [ ]:
from langchain.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy

In [ ]:
# embedding model

embedding_model = HuggingFaceEmbeddings(
    model_name= "intfloat/e5-small-v2",
    multi_process=True,
    encode_kwargs={"normalize_embeddings": True},  # set `True` for cosine similarity
)

In [27]:
# create vector store

knowledge_vector_db = FAISS.from_documents(
    split_text, embedding_model, distance_strategy=DistanceStrategy.COSINE
)

In [28]:
# save vector store locally
knowledge_vector_db.save_local("../data/temp/new_haven_zoning_code_vector_db")

In [29]:
# load vector store
emb_vector_db = FAISS.load_local("../data/temp/new_haven_zoning_code_vector_db", 
                                embedding_model,
                                allow_dangerous_deserialization=True
        )

In [30]:
# retrieve relevant context based on query

# Question based on Section 17. RO Districts: Residence-Office, Page 47 of the PDF
query = "For what purposes can a building be constructed in an RO District?"

retrieved_context = emb_vector_db.similarity_search(query, k=5)

In [31]:
# examine top context
print(retrieved_context[0].page_content)
## Output: Correct chunk identified but chunk size limits the information retrieved

# context size
len(retrieved_context[0].page_content)

offices, and to such other non-residential uses as generally support and harmonize with a high density area of this 
type. The non-residential uses permitted in RO Districts, subject to adequate conditions and safeguards, are 
hereby found and declared to be the only appropriate such uses for such areas. It is hereby found and declared, 
further, that these regulations are necessary to the protection of these areas and that their protection is essential 
to the maintenance of a balanced community of sound residential areas of diverse types.  
All RO Districts are subject to the General Provisions for Residence Districts set forth in Article IV as well as to all 
other provisions of this ordinance.  
Uses permitted. In an RO District a building or other structure may be erected, altered, arranged, designed or 
used, and a lot or structure may be used for any of the following purposes and no other:


909

In [32]:
## Output: Context ranked 1, 2, and 4 did not identify RO district
## Context ranked 3 is correctly about RO districts but is not an exact match to the query
print(retrieved_context[3].page_content)

Created: 2021-04-06 13:09:56 [EST] 
(Supp. No. 27) 
 
Page 49 of 211 
 5 Accessory buildings may extend into side and rear yards. (See General Provisions for Residence Districts).  
Other Requirements: 
Minimum lot area: 7,500 square feet for RH-1 and RO; 5,400 square feet for RH-2.  
Maximum floor area ratio (F.A.R.): 0.5 to 1.7, depending upon buildingcoverage (see text), except for 
RH-2 where F.A.R. is 2.0.  
Maximum buildingcoverage: principal building(s) 25% or less for the RH-1 and RO, (see text), 50% for 
the RH-2, accessory building, 10%.  
Maximum building height: no direct limit, except for zero lot line developments. 
Minimum usable open space: 125 sq. ft. per dwelling unit. 
Minimum parking: For the RH-1 and RO districts, one parking space perdwelling unit. For the RH-2 district, .75 
parking space per dwelling unit, located on the same lot, within 300 feet walking distance or in a multi-lot


In [16]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [ ]:
# create model pipeline

llama_model = pipeline(
    model = model,
    tokenizer = tokenizer,
    task = "text-generation",
    do_sample=True,
    temperature=0.2
)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


ValueError: block_mask was created for block_mask.shape=(1, 1, 8, 8) but got q_len=1 and kv_len=8. As the block mask was created for a larger length than you're using it for, you can either 1. create a new block mask with the correct length, or 2. 'adjust' the existing block mask to the correct length by calling block_mask._adjust(q_len, kv_len). This essentially 'crops' the block mask to the upper left corner, which does not work for all mask_mods!

In [ ]:
# create prompt template

prompt = [
    {
        "role": "system",
        "content": """You are a helpful assistant that has knowledge about the New Haven Zoning Code.
        Based on the context provided, your task is to answer a user's question.
        Only use the information provided in the context to answer the question. 
        If the answer cannot be deduced from the context, then do not answer the questions.""",
    },
    {
        "role": "user",
        "content": """Context:
        {context}
        ---
        Question:
        {query}""",
        },
    ]

prompt_template = tokenizer.apply_chat_template(
    prompt, tokenize=False, add_generation_prompt=True
)

In [ ]:
# ask the model 

input_prompt = prompt_template.format(
    context=retrieved_context[0].page_content,
    query=query
)

model_response = llama_model(input_prompt)[0]["generated_text"]
print(model_response)